# RAG over a PDF

In [1]:
pip -q install pdfplumber sentence-transformers faiss-cpu transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 43.4 MB/s eta 0:00:00


In [2]:
pip install hf_xet

In [3]:

import re
import numpy as np
import pdfplumber
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Load PDF and extract text

In [4]:

PDF_PATH = "/content/aru_regulations.pdf"

def extract_text_from_pdf(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            txt = page.extract_text() or ""
            # Keep explicit page markers for later citation
            pages.append(f"\n\n--- PAGE {i+1} ---\n{txt}")
    return "\n".join(pages)

raw_text = extract_text_from_pdf(PDF_PATH)
print("Extracted characters:", len(raw_text))
print(raw_text[:600])


Extracted characters: 410260


--- PAGE 1 ---
Academic
Regulations
Eighteenth Edition
September 2025
aru.ac.uk/academicregs


--- PAGE 2 ---
Academic
Regulations
Eighteenth Edition
September 2025
aru.ac.uk/academicregs


--- PAGE 3 ---
CONTENTS
Introduction 5-6
Section
1. Foreword 7
(A) Introduction 7
(B) Senate Codes of Practice 8
2. Anglia Ruskin University Awards 10
(A) List of ARU Awards 10
(B) Definitions 13
(C) General Principles of the Undergraduate and Postgraduate Taught 24
Curriculum
(D) Curriculum Structure 27
(E) Academic Standard of ARU Awards 30
3. Curriculum Structures and Duration of Study 55
(A) Design Pr


#  Light cleaning (don’t over-clean regulations)

In [5]:

def clean_text(t):
    t = t.replace("\x00", "")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

text = clean_text(raw_text)
print("Cleaned characters:", len(text))


Cleaned characters: 410038


In [6]:
print(text[:600])

--- PAGE 1 ---
Academic
Regulations
Eighteenth Edition
September 2025
aru.ac.uk/academicregs

--- PAGE 2 ---
Academic
Regulations
Eighteenth Edition
September 2025
aru.ac.uk/academicregs

--- PAGE 3 ---
CONTENTS
Introduction 5-6
Section
1. Foreword 7
(A) Introduction 7
(B) Senate Codes of Practice 8
2. Anglia Ruskin University Awards 10
(A) List of ARU Awards 10
(B) Definitions 13
(C) General Principles of the Undergraduate and Postgraduate Taught 24
Curriculum
(D) Curriculum Structure 27
(E) Academic Standard of ARU Awards 30
3. Curriculum Structures and Duration of Study 55
(A) Design Princi


# Chunking with overlap (simple char-based chunking)
### Note: Token-based chunking is better; this is simple and works for demos.)

In [7]:

def chunk_text(t, chunk_size=1200, overlap=250):
    chunks = []
    start = 0
    n = len(t)
    while start < n:
        end = min(start + chunk_size, n)
        chunks.append(t[start:end])
        if end >= n:
            break
        start = max(0, end - overlap)
    return chunks

chunks_text = chunk_text(text, chunk_size=1200, overlap=250)
print("Number of chunks:", len(chunks_text))
print("Example chunk:\n", chunks_text[0][:500])

Number of chunks: 432
Example chunk:
 --- PAGE 1 ---
Academic
Regulations
Eighteenth Edition
September 2025
aru.ac.uk/academicregs

--- PAGE 2 ---
Academic
Regulations
Eighteenth Edition
September 2025
aru.ac.uk/academicregs

--- PAGE 3 ---
CONTENTS
Introduction 5-6
Section
1. Foreword 7
(A) Introduction 7
(B) Senate Codes of Practice 8
2. Anglia Ruskin University Awards 10
(A) List of ARU Awards 10
(B) Definitions 13
(C) General Principles of the Undergraduate and Postgraduate Taught 24
Curriculum
(D) Curriculum Structure 27
(E) Ac


#   Build per-chunk metadata (dicts, not classes)

In [8]:

def first_page_marker(chunk_str):
    m = re.search(r"--- PAGE (\d+) ---", chunk_str)
    return int(m.group(1)) if m else -1

chunks = []
for i, ch in enumerate(chunks_text):
    chunks.append({
        "chunk_id": i,
        "page_hint": first_page_marker(ch),
        "text": ch
    })


In [9]:
chunks[1]

{'chunk_id': 1,
 'page_hint': 4,
 'text': 'nts 72\n1\n\n--- PAGE 4 ---\n(E) Applicants for Whom English is not the First Language 75\n(F) Accreditation of Prior Learning 77\n(G) Applicants with a Criminal Conviction 81\n(H) Disabled Applicants and Applicants with Specific Learning Difficulties 83\n(J) Fraudulent Applications 84\n5. Student Conduct, Rights and Responsibilities 85\n(A) Student Conduct 85\n(B) Student Rights 85\n(C) Student Responsibilities 86\n6. Assessment 88\n(A) Introduction 88\n(B) Purpose of Assessment 88\n(C) Principles 88\n(D) Equity and Clarity in Assessment 90\n(E) Objectivity and Independence in Assessment 90\n(F) Language of Assessment 91\n(G) Ethical Approval for Research 91\n(H) Module Assessment 93\n(J) Submission of Work for Assessment 99\n(K) Short Term Extensions 102\n(L) Long Term Extensions 104\n(M) Exceeding Word Limits 105\n(N) Module Re-assessment: Number of Attempts, Form, Timing and 105\nModule Result\n(P) Retaking or Replacing a Failed Module Aft

# Embeddings

In [10]:

embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

embeddings = embedder.encode(
    [c["text"] for c in chunks],
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
print("Embeddings shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Embeddings shape: (432, 384)


In [11]:
embeddings[1]

array([-5.66980429e-02,  2.24095881e-02, -4.71604019e-02, -2.76990347e-02,
       -4.99996766e-02,  6.51281327e-02,  2.54266169e-02, -1.10676102e-02,
       -8.96781161e-02,  4.41798344e-02,  7.01907873e-02,  3.83923249e-03,
        3.72241139e-02,  3.09017655e-02, -8.29277709e-02,  6.64252508e-03,
       -3.57755683e-02, -1.12595260e-02, -3.65007818e-02,  1.07187880e-02,
        3.13210674e-02,  5.41562811e-02, -4.20917161e-02, -3.64818014e-02,
        6.16077986e-03, -4.50153137e-03, -4.12569642e-02,  4.11424041e-03,
        6.33196160e-02, -5.87728061e-02, -2.94802729e-02,  3.70785706e-02,
        2.56252382e-02,  1.86110660e-02,  6.74747825e-02, -1.66895334e-02,
        2.08472274e-02, -1.02457060e-02, -3.18048485e-02, -3.37452628e-02,
       -2.24297829e-02, -9.31148902e-02, -6.18406478e-03, -2.69488264e-02,
        3.41539681e-02, -2.36653350e-02, -1.98721439e-02, -4.83598411e-02,
       -5.97541109e-02,  5.80986924e-02, -4.31612320e-02,  1.81112643e-02,
       -2.83264555e-03,  

#  Build FAISS index
### Using cosine similarity via normalized vectors + inner product index

In [12]:

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print("FAISS index size:", index.ntotal)

FAISS index size: 432


#  Retrieval

In [13]:

def retrieve(query, top_k=5):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, ids = index.search(q_emb, top_k)

    results = []
    for idx, score in zip(ids[0], scores[0]):
        if idx == -1:
            continue
        c = chunks[int(idx)]
        results.append((c, float(score)))
    return results


#  Generator (local)

It uses a local HF model __(google/flan-t5-base)__ so you don’t need an API key. __flan-t5-base__ is small; it’s fine for a demo. If you want higher answer quality, swap the generator to a stronger local model (e.g., via Ollama) or your institution’s approved hosted LLM.

In [15]:
# It uses a local HF model (google/flan-t5-base) so you don’t need an API key.
gen = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=256
)

SYSTEM_RULES = (
    "You are an assistant answering questions about ARU academic regulations.\n"
    "Use ONLY the provided context. If the context is insufficient, say: "
    "'I couldn't find that in the provided document.'\n"
    "Be concise and accurate.\n"
)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCa

# Augmentation

__build_prompt__ is responsible for the augmentation step in RAG.
It takes the retrieved document chunks and injects them into a structured prompt that the LLM can reason over.

In [16]:
def build_prompt(query, retrieved_pairs):
    # retrieved_pairs: list of (chunk_dict, score)
    blocks = []
    for c, score in retrieved_pairs:
        page = c["page_hint"] if c["page_hint"] != -1 else "Unknown"
        blocks.append(
            f"[Source: page {page}, chunk {c['chunk_id']}, score {score:.3f}]\n{c['text']}"
        )

    context = "\n\n".join(blocks)

    prompt = (
        f"{SYSTEM_RULES}\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION:\n{query}\n\n"
        f"ANSWER:"
    )
    return prompt

# RAG inference pipeline creation

In [17]:
def rag_answer(query, top_k=6):
    retrieved = retrieve(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)
    answer = gen(prompt)[0]["generated_text"].strip()

    # Return top sources for traceability
    sources = []
    for c, score in retrieved:
        sources.append({
            "page_hint": c["page_hint"],
            "chunk_id": c["chunk_id"],
            "score": score,
            "preview": c["text"][:220].replace("\n", " ")
        })

    return {"query": query, "answer": answer, "sources": sources}



#   Testing

In [18]:

q = "What does the regulation say about academic misconduct or assessment offences?"
out = rag_answer(q, top_k=6)

print("Q:", out["query"])
print("\nAnswer:\n", out["answer"])
print("\nTop sources:")
for s in out["sources"][:3]:
    print(f"- page_hint={s['page_hint']} chunk={s['chunk_id']} score={s['score']:.3f} :: {s['preview']} ...")


# Cell 11) Interactive loop (optional)
while True:
    user_q = input("\nAsk a question (type 'exit' to stop): ").strip()
    if user_q.lower() in ("exit", "quit"):
        break
    out = rag_answer(user_q, top_k=6)
    print("\nAnswer:\n", out["answer"])
    print("\nSources:")
    for s in out["sources"][:3]:
        print(f"- page_hint={s['page_hint']} chunk={s['chunk_id']} score={s['score']:.3f} :: {s['preview']} ...")

Token indices sequence length is longer than the specified maximum sequence length for this model (1675 > 512). Running this sequence through the model will result in indexing errors


Q: What does the regulation say about academic misconduct or assessment offences?

Answer:
 You are an assistant answering questions about ARU academic regulations.
Use ONLY the provided context. If the context is insufficient, say: 'I couldn't find that in the provided document.'
Be concise and accurate.


CONTEXT:
[Source: page 181, chunk 374, score 0.732]
academic
misconduct to which the case has been assigned (see Regulation 10.23 above) and are
specified in Regulation 10.66 below.
10.55 Where a confirmed case under categories A and B is the first incidence of academic
misconduct for an individual student, and completion of the on-line Academic Integrity
Course has been confirmed, the default penalty assigned to the category of academic
misconduct is reduced to the next immediate lower category (e.g. the penalty for a category
B case is amended to be the penalty assigned to category A case). For a category A case,
the outcome of a reduced penalty is for the piece of work which is s